In [ ]:
import BioSimSpace as BSS

import pandas as pd

from pathlib import Path

import re

# on this occasion we will not be using a node, instead we will use a function
from nodes.fep_prep import prepare_fep_free, prepare_fep_bound
from dask.distributed import Client, LocalCluster, wait, as_completed

In [ ]:
def get_ligand_names(filename):
    p = Path(filename).stem
    p = p.split("_")[0]
    return p

In [ ]:
# we will grab the engine and runtime from our protocol file
config = {}
# read config
with open("output_setup/protocol.dat", "r") as file:
    for line in file:
        key, value = line.split("=")
        config[str(key).strip()] = str(value).strip().replace("*", "")

In [ ]:
runtime = BSS.Types.Time(config["sampling"]).nanoseconds()

In [ ]:
# read the network file to find our perturbations
df_network = pd.read_csv(
    "output_setup/network.dat",
    sep="\s+",
    names=["lig1", "lig2", "num_windows", "lambda_vals", "engine"],
)

In [ ]:
free_folder = Path("equilibrated_free_systems")
free_files = list(free_folder.glob("*.prm7")) + list(free_folder.glob("*.rst7"))

bound_folder = Path("equilibrated_bound_systems")
bound_files = list(bound_folder.glob("*.prm7")) + list(bound_folder.glob("*.rst7"))

In [ ]:
# files for free legs
ligand_names_locations_free = {}
for files in free_files:
    name = get_ligand_names(files)
    if name in ligand_names_locations_free.keys():
        ligand_names_locations_free[name].append(str(files.absolute()))
    else:
        ligand_names_locations_free[name] = [str(files.absolute())]

# files for bound legs
ligand_names_locations_bound = {}
for files in bound_files:
    name = get_ligand_names(files)
    if name in ligand_names_locations_bound.keys():
        ligand_names_locations_bound[name].append(str(files.absolute()))
    else:
        ligand_names_locations_bound[name] = [str(files.absolute())]

In [ ]:
inputs_for_free_runs = []
# now we loop over the given perturbations and create inputs for our prep function
for row in df_network.iterrows():
    row = row[1]
    lig1 = row["lig1"]
    lig2 = row["lig2"]
    if (lig1 not in ligand_names_locations_free.keys()) or (
        lig2 not in ligand_names_locations_free.keys()
    ):
        print(f"ligands not found in equilibrated free systems ({lig1} or {lig2})")
        continue
    lambds = [float(i) for i in row["lambda_vals"].strip().split(",")]
    # engine= config["engine"]
    # settings for testing
    engine = "SOMD"
    runtime = "1ps"
    inp = {
        "ligand1_name": lig1,
        "ligand2_name": lig2,
        "ligand1_files": ligand_names_locations_free[lig1],
        "ligand2_files": ligand_names_locations_free[lig2],
        "output_location": Path(f"production/{engine}/").absolute(),
        "num_lambda": len(lambds),
        "lambda_values": lambds,
        "md_engine": engine,
        "runtime": runtime,
    }
    inputs_for_free_runs.append(inp)

In [ ]:
inputs_for_bound_runs = []
# now we loop over the given perturbations and create inputs for our prep function
for row in df_network.iterrows():
    row = row[1]
    lig1 = row["lig1"]
    lig2 = row["lig2"]
    if (lig1 not in ligand_names_locations_bound.keys()) or (
        lig2 not in ligand_names_locations_bound.keys()
    ):
        print(f"ligands not found in equilibrated bound systems ({lig1} or {lig2})")
        continue
    lambds = [float(i) for i in row["lambda_vals"].strip().split(",")]
    # engine= config["engine"]
    # settings for testing
    engine = "SOMD"
    runtime = "1ps"
    inp = {
        "ligand1_name": lig1,
        "ligand2_name": lig2,
        "ligand1_files": ligand_names_locations_bound[lig1],
        "ligand2_files": ligand_names_locations_bound[lig2],
        "output_location": Path(f"production/{engine}/").absolute(),
        "num_lambda": len(lambds),
        "lambda_values": lambds,
        "md_engine": engine,
        "runtime": runtime,
    }
    inputs_for_bound_runs.append(inp)

In [ ]:
# Now we can create a localcluster to setup our simulation network.
# If using `setup_only=False` it is strongly recommended to use HPC resources as running the full network will be very computationally intensive
cluster = LocalCluster(
    n_workers=4,
    memory_limit="5000MB",
    threads_per_worker=5,
    # We will use CPU_task to ensure that only a single parameterisation/solvation is performed per worker
    resources={"CPU_task": 1},
)

In [ ]:
client = Client(cluster)
# node_plugin = SetNodeDir(node_path)
# client.register_plugin(node_plugin, name="node-setup")
client

In [ ]:
futures = [
    client.submit(prepare_fep_free, **inp, resources={"CPU_task": 1})
    for inp in inputs_for_free_runs
]

In [ ]:
# as a workaround for uncontrolled memory usage in dask, we will run our prep in chunks of len(num workers)
for i in range(0, len(inputs_for_free_runs), len(cluster.workers)):
    chunk_inputs = inputs_for_free_runs[i:i+4]
    f = [client.submit(prepare_fep_free, **inp) for inp in chunk_inputs]
    _ = wait(f)
    client.restart()

In [ ]:
# as a workaround for uncontrolled memory usage in dask, we will run our prep in chunks of len(num workers)
for i in range(0, len(inputs_for_bound_runs), len(cluster.workers)):
    chunk_inputs = inputs_for_bound_runs[i:i+4]
    f = [client.submit(prepare_fep_bound, **inp) for inp in chunk_inputs]
    _ = wait(f)
    client.restart()